# Notebook 1: Exploratory Data Analysis

Framingham Heart Study — 10-year CVD risk prediction

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys; sys.path.append('..')
from src.preprocess import load_data, summarize_missing

sns.set_theme(style='whitegrid')
df = load_data('../data/framingham.csv')
df.head()

In [ ]:
df.info()
print('\nMissing values:')
print(summarize_missing(df))

In [ ]:
# Class balance (note: imbalanced target)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
counts = df['TenYearCHD'].value_counts().sort_index()
counts.plot(kind='bar', ax=ax[0], color=['steelblue', 'tomato'])
ax[0].set_title('Target Class Distribution')
ax[0].set_xticklabels(['No CVD', 'CVD'], rotation=0)

pos_rate = df['TenYearCHD'].mean()
ax[0].text(0.5, 0.9, f"Positive rate ≈ {pos_rate:.3f}\n(Accuracy can be misleading)",
           transform=ax[0].transAxes, ha='center', va='top')

# Age distribution by target
df.groupby('TenYearCHD')['age'].plot(kind='hist', ax=ax[1], alpha=0.6, bins=20, legend=True)
ax[1].set_title('Age Distribution by CVD Label')
plt.tight_layout()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 9))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.tight_layout()

In [ ]:
# Feature distributions
numeric_cols = df.select_dtypes(include=np.number).columns.drop('TenYearCHD')
df[numeric_cols].hist(bins=20, figsize=(14, 10))
plt.suptitle('Feature Distributions', y=1.01)
plt.tight_layout()

# Variance Inflation Factor (VIF) for multicollinearity diagnosis
# Compute VIF on the TRAIN split only to avoid leakage.
from sklearn.model_selection import train_test_split
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_raw = df.drop(columns=['TenYearCHD'])
y_raw = df['TenYearCHD'].values
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw
)

# Light imputation for VIF computation (fit on train only)
imputer = IterativeImputer(random_state=42, max_iter=20, sample_posterior=False, initial_strategy='median')
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train_raw), columns=X_train_raw.columns)

# VIF (higher -> more multicollinearity). Rules of thumb: >5 moderate, >10 severe.
vif = pd.DataFrame({
    'feature': X_train_imp.columns,
    'vif': [variance_inflation_factor(X_train_imp.values, i) for i in range(X_train_imp.shape[1])]
}).sort_values('vif', ascending=False)

display(vif.head(15))